In [2]:
# HuggingFace Transformersのインストール
# Google Colabでは最初から入っていることが多いが、
# 最新版にするためにインストールを実行

!pip install transformers torch torchvision

# バージョン確認
import transformers
print(f"transformers version: {transformers.__version__}")

transformers version: 5.16.1


In [ ]:
from pathlib import Path
from io import BytesIO

import requests
from PIL import Image
from IPython.display import display
from transformers import pipeline

# 1. 保存先と、取得する COCO val2017 の画像を指定
save_dir = Path("coco_samples")
save_dir.mkdir(exist_ok=True)

image_ids = [
    324158,  # 犬が写っている画像（人物なども写っています）
    39769,   # 猫が写っている画像
]

# COCO 画像が置かれている S3 バケットの HTTPS URL
base_url = "https://s3.amazonaws.com/images.cocodataset.org/val2017"

# 2. 画像をダウンロード
image_paths = []

for image_id in image_ids:
    filename = f"{image_id:012d}.jpg"
    path = save_dir / filename

    if not path.exists():
        url = f"{base_url}/{filename}"
        response = requests.get(url, timeout=60)
        response.raise_for_status()

        # 画像として読み込めることを確認してから保存
        with Image.open(BytesIO(response.content)) as img:
            img.verify()

        path.write_bytes(response.content)
        print(f"Downloaded: {path}")
    else:
        print(f"Already exists: {path}")

    image_paths.append(path)

# 3. 画像分類パイプラインを作成
classifier = pipeline(
    "image-classification",
    model="google/vit-base-patch16-224",
)

# 4. 各画像を表示して分類
for path in image_paths:
    with Image.open(path) as img:
        image = img.convert("RGB")

    print(f"\n===== {path.name} =====")

    preview = image.copy()
    preview.thumbnail((480, 480))
    display(preview)

    results = classifier(image, top_k=5)

    print("Top 5 Predictions:")
    for result in results:
        print(f"  {result['label']}: {result['score'] * 100:.2f}%")

Downloaded: coco_samples/000000324158.jpg
Downloaded: coco_samples/000000039769.jpg


In [ ]:
# ===================================================
# 詳細な制御：processor + model
# ===================================================

from transformers import ViTImageProcessor, ViTForImageClassification
from PIL import Image
import torch

# 1. モデルとプロセッサをロード
# google/vit-large-patch16-224:
#   – google: 提供元
#   – vit-large: モデルサイズ（large = 307M パラメータ）
#   – patch16: パッチサイズ（16×16）
#   – 224: 入力画像サイズ（224×224）
model_name = "google/vit-large-patch16-224"

# ViTImageProcessor: 画像の前処理を担当
# – リサイズ（224×224）
# – 正規化（ImageNetの平均・標準偏差）
# – テンソル変換
processor = ViTImageProcessor.from_pretrained(model_name)

# ViTForImageClassification: 画像分類モデル
# – Transformer Encoder + 分類ヘッド
# – ImageNetの1000クラス分類用
model = ViTForImageClassification.from_pretrained(model_name)

# 評価モードに設定(Dropout無効化、BatchNormの統計値固定)
model.eval()

# GPUが使える場合はGPUに転送
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(f"モデル: {model_name}")
print(f"クラス数: {model.config.num_labels}")
print(f"デバイス: {device}")

# ===================================================
# 2. 画像の読み込みと前処理
# ===================================================
# 画像を読み込む
# 画像はRGBに変換しておく（ViTはRGB前提）

# 1. 保存先と、取得する COCO val2017 の画像を指定
save_dir = Path("coco_samples")
save_dir.mkdir(exist_ok=True)

image_ids = [
    324158,  # 犬が写っている画像（人物なども写っています）
    39769,   # 猫が写っている画像
]

# COCO 画像が置かれている S3 バケットの HTTPS URL
base_url = "https://s3.amazonaws.com/images.cocodataset.org/val2017"

# 2. 画像をダウンロード
image_paths = []

for image_id in image_ids:
    filename = f"{image_id:012d}.jpg"
    path = save_dir / filename

    if not path.exists():
        url = f"{base_url}/{filename}"
        response = requests.get(url, timeout=60)
        response.raise_for_status()

        # 画像として読み込めることを確認してから保存
        with Image.open(BytesIO(response.content)) as img:
            img.verify()

        image = Image.open(BytesIO(response.content)).convert("RGB")

        path.write_bytes(response.content)
        print(f"Downloaded: {path}")
    else:
        print(f"Already exists: {path}")

    image_paths.append(path)

# 前処理を実行
# ViTImageProcessorは、画像をリサイズ、正規化、テンソル変換してくれる
# – リサイズ: 224×224
# – 正規化: (pixel – mean) / std
# – テンソル化: PILイメージ → PyTorchテンソル
# return_tensors="pt": PyTorchテンソルを返す
inputs = processor(images=[Image.open(path).convert("RGB") for path in image_paths], return_tensors="pt")

#GPUに転送
inputs = {k: v.to(device) for k, v in inputs.items()}

print(f"入力テンソルの形状: {inputs['pixel_values'].shape}")  # (batch_size, 3, 224, 224)
print(f"入力テンソルのデバイス: {inputs['pixel_values'].device}")  # cuda or cpu



In [ ]:
# ===================================================
# 3. 推論
# ===================================================

# torch.no_grad(): 勾配計算を無効化（推論時は不要）
# メモリ使用量を削減し、計算速度を向上させる
with torch.no_grad():
    # モデルに入力テンソルを渡して推論
    # **inputs: 辞書を展開してキーワード引数として渡す
    outputs = model(**inputs)

    # 出力はlogits（未正規化のスコア）
    # outputs.logits: 分類ヘッドの出力 (batch_size, num_labels) softmaxを適用する前の値]
    # (batch_size, num_labels) = (len(image_paths), model.config.num_labels)
    logits = outputs.logits

    # ソフトマックスで確率に変換
    # dim=-1: 最後の次元（クラス数）に沿って計算
    probabilities = torch.nn.functional.softmax(logits, dim=-1)

    # 上位5クラスを取得
    top5_prob, top5_indices = torch.topk(probabilities, k=5, dim=-1)

    print(f"出力テンソルの形状: {probabilities.shape}")  # (batch_size, num_labels)
    print(f"上位5クラスの確率: {top5_prob}")
    print(f"上位5クラスのインデックス: {top5_indices}")
    print(f"確率の合計: {probabilities.sum(dim=-1)}")  # 1.0に近い値になるはず

    #各画像を表示して分類結果を出力
    print("分類結果:")

    for path in image_paths:
        with Image.open(path) as img:
            image = img.convert("RGB")

        print(f"\n===== {path.name} =====")

        preview = image.copy()
        preview.thumbnail((480, 480))
        display(preview)

    for i, path in enumerate(image_paths):
        print(f"\n===== {path.name} =====")
        for j in range(5):
            class_idx = top5_indices[i, j].item()
            class_prob = top5_prob[i, j].item()
            class_label = model.config.id2label[class_idx]
            print(f"  {class_label}: {class_prob * 100:.2f}%")
